## Import Libraries

In [14]:
import re, os, sys
import numpy as np
import pandas as pd
import unicodedata
import matplotlib.pyplot as plt
from tqdm import tqdm
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import nltk
from nltk.corpus import stopwords as nltk_stopwords
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, word_tokenize
from nltk.corpus import wordnet
from nltk.sentiment import SentimentIntensityAnalyzer
from scipy.sparse import vstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer



In [ ]:
# Set default renderer for Plotly to VSCode
pio.renderers.default = "vscode"

## Setup

In [16]:
for pkg in [
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng",  # for NLTK >= 3.8
    "punkt",
    "punkt_tab",
    "stopwords",
    "wordnet",
    "omw-1.4"
]:
    nltk.download(pkg, quiet=False)

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/subhadeepdebnath/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/subhadeepdebnath/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/subhadeepdebnath/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/subhadeepdebnath/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/subhadeepdebnath/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/subhadeepdebnath/nltk_data...
[nltk_data]   Package wordnet is 

## Loading dataset

In [17]:
imdb_data = pd.read_csv('/Users/subhadeepdebnath/Developer/genai/experiments/nlp_playground/data/IMDB_Dataset.csv')
imdb_data.head()

,review,sentiment
0,"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<b...",positive
1,"A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomf...",positive
2,"I thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air conditioned theater and watching a light-hearted comedy. Th...",positive
3,Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his parents are fighting all the time.<br /><br />This movie is...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is a visually stunning film to watch. Mr. Mattei offers us a vivid portrait about human relations. This is a mov...",positive


In [18]:
print(f"Dataset shape: {imdb_data.shape}")
print(f"Dataset columns: {imdb_data.columns.tolist()}")
print("", imdb_data.info())

Dataset shape: (50000, 2)
Dataset columns: ['review', 'sentiment']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB
 None


In [19]:
SAMPLE_SIZE = None        # Set to None to use full dataset
TFIDF_MAX_FEATURES = 10000 # Max features for TF-IDF Vectorizer
N_TOPICS = 10        # Number of topics for LDA
N_TOP_TERMS = 12     # Number of top terms to display per topic
RANDOM_STATE = 42      # Random state for reproducibility

In [20]:
imdb_data["review"] = imdb_data["review"].astype(str)
imdb_data["sentiment"] = imdb_data["sentiment"].astype(str)

In [21]:
vc = (imdb_data["sentiment"]
      .value_counts()
      .rename_axis("sentiment")
      .reset_index(name="count"))

fig = px.bar(
    vc, x="sentiment", y="count",
    labels={"sentiment":"Sentiment", "count":"Count"},
    title="IMDb Sentiment Distribution"
)
fig.update_layout(yaxis=dict(gridcolor="rgba(0,0,0,0.1)"))

fig.show()

# Review length (in tokens) — before cleaning
imdb_data["review_len"] = imdb_data["review"].str.split().apply(len)
fig = px.histogram(
    imdb_data, x="review_len", nbins=60,
    title="Distribution of Raw Review Lengths (Tokens)"
)
fig.update_layout(bargap=0.02)
fig.show()


### Data Cleaning

In [22]:
def _wn_pos(tag):
    """Map Penn Treebank tag -> WordNet POS."""
    if tag.startswith('J'): return wordnet.ADJ
    if tag.startswith('V'): return wordnet.VERB
    if tag.startswith('N'): return wordnet.NOUN
    if tag.startswith('R'): return wordnet.ADV
    return wordnet.NOUN  # fallback

In [23]:
def data_cleaning(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+|#\w+", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.lower()
    text = re.sub(r"[^a-z\s']", " ", text)    # ✅ keep only English letters + apostrophe
    text = re.sub(r"\s+", " ", text).strip()
    return text


## Data Preparation

In [24]:
def data_prep(text: str, keep_negations: bool = True, verbose: bool = False):
    tokens = word_tokenize(text)
    if verbose:
        print("\nTokens after Tokenization:", tokens)

    sw = set(nltk_stopwords.words("english"))
    if keep_negations:
        for w in ["no", "not", "nor", "n't"]:
            if w in sw: sw.remove(w)

    tokens = [t for t in tokens if (t not in sw and t.isalpha()) or ("'" in t and keep_negations)]

    lem = WordNetLemmatizer()
    tagged = pos_tag(tokens)
    lemmas = [lem.lemmatize(tok, _wn_pos(pos)) for tok, pos in tagged]
    return lemmas

In [25]:
tqdm.pandas(desc="Data Cleaning & Preparation")

# Avoid recompute if already present
if "clean" not in imdb_data.columns:
    imdb_data["clean"]  = imdb_data["review"].progress_apply(data_cleaning)

if "tokens" not in imdb_data.columns:
    imdb_data["tokens"] = imdb_data["clean"].progress_apply(lambda t: data_prep(t, keep_negations=True))

if "prep" not in imdb_data.columns:
    imdb_data["prep"]   = imdb_data["tokens"].apply(lambda toks: " ".join(toks))

# Peek a few rows
pd.set_option("display.max_colwidth", 160)
imdb_data[["review", "prep", "sentiment"]].sample(5, random_state=RANDOM_STATE)


Data Cleaning & Preparation: 100%|██████████| 50000/50000 [02:44<00:00, 303.36it/s]


,review,prep,sentiment
33553,"I really liked this Summerslam due to the look of the arena, the curtains and just the look overall was interesting to me for some reason. Anyways, this cou...",really liked summerslam due look arena curtain look overall interesting reason anyways could one best summerslam 's ever wwf n't lex luger main event yokozu...,positive
9427,Not many television shows appeal to quite as many different kinds of fans like Farscape does...I know youngsters and 30/40+ years old;fans both Male and Fem...,not many television show appeal quite many different kind fan like farscape know youngster year old fan male female many different country think adore v min...,positive
199,The film quickly gets to a major chase scene with ever increasing destruction. The first really bad thing is the guy hijacking Steven Seagal would have been...,film quickly get major chase scene ever increase destruction first really bad thing guy hijack steven seagal would beat pulp seagal 's driving probably woul...,negative
12447,Jane Austen would definitely approve of this one!<br /><br />Gwyneth Paltrow does an awesome job capturing the attitude of Emma. She is funny without being ...,jane austen would definitely approve one gwyneth paltrow awesome job capture attitude emma funny without excessively silly yet elegant put convince british ...,positive
39489,"Expectations were somewhat high for me when I went to see this movie, after all I thought Steve Carell could do no wrong coming off of great movies like Anc...",expectation somewhat high go see movie think steve carell could no wrong come great movie like anchorman year old virgin little miss sunshine boy wrong 'll ...,negative


## Feature Creation 

### TF-IDF Vectoriser

In [26]:
# TF-IDF for supervised models & similarity
tfidf_vectorizer = TfidfVectorizer(max_features=TFIDF_MAX_FEATURES, stop_words="english")
X_tfidf = tfidf_vectorizer.fit_transform(imdb_data["prep"])

### Bag of Words (BoW)

In [27]:
# BoW for LDA topic modeling (LDA expects counts)
count_vectorizer = CountVectorizer(max_features=TFIDF_MAX_FEATURES, stop_words="english")
X_bow = count_vectorizer.fit_transform(imdb_data["prep"])

In [28]:
y_true = imdb_data["sentiment"].map({"positive":1, "negative":0}).values

print("TF-IDF shape:", X_tfidf.shape, "| BoW shape:", X_bow.shape)

TF-IDF shape: (50000, 10000) | BoW shape: (50000, 10000)


#### Sentiment Analysis (VADER)

In [29]:
sia = SentimentIntensityAnalyzer()

def vader_label(text, pos=0.05, neg=-0.05):
    s = sia.polarity_scores(text)["compound"]
    if s >= pos: return 1   # positive
    if s <= neg: return 0   # negative
    return None             # neutral/uncertain

pseudo_y = [vader_label(doc) for doc in imdb_data["prep"]]
y_filled = [0 if y is None else y for y in pseudo_y]

# drop neutrals if you want strict binary
X_keep = []
y_keep = []
for x, y in zip(X_tfidf, y_filled):
    if y is not None:
        X_keep.append(x)
        y_keep.append(y)

X_keep = vstack(X_keep)
y_keep = np.array(y_keep)


## Modelling


### Train-Test Split

In [30]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y_true, test_size=0.2, stratify=y_true, random_state=RANDOM_STATE
)
print("Train shape:", X_train.shape, "| Test shape:", X_test.shape) 

Train shape: (40000, 10000) | Test shape: (10000, 10000)


### Text Classification

#### Naive Bayes

In [31]:
nb = MultinomialNB()
nb.fit(X_train, y_train)
pred_nb = nb.predict(X_test)

print(classification_report(y_test, pred_nb))



              precision    recall  f1-score   support

           0       0.86      0.85      0.85      5000
           1       0.85      0.86      0.85      5000

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000



#### Note:
- Using sentiment col of dataset for model training

In [32]:
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y_keep, test_size=0.2, random_state=42)
clf = MultinomialNB()
clf.fit(X_train, y_train)
pred = clf.predict(X_test)

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.78      0.48      0.60      3453
           1       0.77      0.93      0.84      6547

    accuracy                           0.78     10000
   macro avg       0.78      0.71      0.72     10000
weighted avg       0.78      0.78      0.76     10000



#### Note:
- Using sentiment vals using sia and vader for model training

#### Logistic Regression

In [33]:
lr = LogisticRegression(max_iter=1000, n_jobs=-1)
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)
proba_lr = lr.predict_proba(X_test)[:,1]
print("Logistic Regression")
print(classification_report(y_test, pred_lr))


Logistic Regression
              precision    recall  f1-score   support

           0       0.83      0.71      0.77      3453
           1       0.86      0.92      0.89      6547

    accuracy                           0.85     10000
   macro avg       0.85      0.82      0.83     10000
weighted avg       0.85      0.85      0.85     10000



In [34]:
# ROC curve for LR
fpr, tpr, _ = roc_curve(y_test, proba_lr)
roc_auc = auc(fpr, tpr)
fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", name=f"AUC={roc_auc:.3f}"))
fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line=dict(dash="dash"))
fig.update_layout(title="ROC Curve — Logistic Regression (TF-IDF)",
                  xaxis_title="False Positive Rate", yaxis_title="True Positive Rate")
fig.show()

In [35]:
#  Confusion matrices (Plotly heatmaps)
def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    fig = go.Figure(data=go.Heatmap(
        z=cm, x=["pred 0","pred 1"], y=["true 0","true 1"],
        text=cm, texttemplate="%{text}", colorscale="Blues"
    ))
    fig.update_layout(title=title, xaxis_title="Predicted", yaxis_title="True")
    fig.show()

plot_cm(y_test, pred_nb, "Confusion Matrix — MultinomialNB (TF-IDF)")
plot_cm(y_test, pred_lr, "Confusion Matrix — Logistic Regression (TF-IDF)")


In [36]:
feature_names = np.array(tfidf_vectorizer.get_feature_names_out())
coefs = lr.coef_[0]

topN = 20
top_pos_idx = np.argsort(coefs)[-topN:]
top_neg_idx = np.argsort(coefs)[:topN]

fig = make_subplots(rows=1, cols=2, subplot_titles=("Top Positive Terms", "Top Negative Terms"))

fig.add_trace(
    go.Bar(x=coefs[top_pos_idx], y=feature_names[top_pos_idx], orientation="h", name="Positive"),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=coefs[top_neg_idx], y=feature_names[top_neg_idx], orientation="h", name="Negative"),
    row=1, col=2
)
fig.update_layout(title="Most Influential Terms (Logistic Regression coefficients)", showlegend=False, height=600)
fig.update_yaxes(categoryorder="array", categoryarray=feature_names[top_pos_idx], row=1, col=1)
fig.update_yaxes(categoryorder="array", categoryarray=feature_names[top_neg_idx], row=1, col=2)
fig.show()



### K-Means Clustering

In [37]:
kmeans = KMeans(n_clusters=2, n_init="auto", random_state=RANDOM_STATE)
kmeans.fit(X_tfidf)
clusters = kmeans.labels_
imdb_data["cluster"] = clusters

# Compare clusters to true labels
print("Adjusted Rand Index (clusters vs sentiment):", adjusted_rand_score(y_true, clusters))

# Top terms per cluster
centers = kmeans.cluster_centers_
terms = np.array(tfidf_vectorizer.get_feature_names_out())

fig = make_subplots(rows=1, cols=2, subplot_titles=("Cluster 0 — Top Terms", "Cluster 1 — Top Terms"))
for c in [0,1]:
    top_idx = centers[c].argsort()[-N_TOP_TERMS:]
    fig.add_trace(go.Bar(x=centers[c][top_idx], y=terms[top_idx], orientation="h"), row=1, col=c+1)
fig.update_layout(title="KMeans — Top Terms per Cluster", showlegend=False, height=600)
fig.show()


Adjusted Rand Index (clusters vs sentiment): 0.018578126842847584



### LDA

In [38]:
lda = LatentDirichletAllocation(
    n_components=N_TOPICS, 
    learning_method="batch",
    max_iter=10, 
    verbose=1,
    random_state=RANDOM_STATE
)
lda.fit(X_bow)

bow_terms = np.array(count_vectorizer.get_feature_names_out())

rows = int(np.ceil(N_TOPICS / 2))
cols = 2 if N_TOPICS > 1 else 1
fig = make_subplots(rows=rows, cols=cols, subplot_titles=[f"Topic {i+1}" for i in range(N_TOPICS)])

for t in tqdm(range(N_TOPICS), desc="LDA Topics"):
    topic = lda.components_[t]
    top_idx = topic.argsort()[-N_TOP_TERMS:]
    # Optional: largest at top (reverse)
    top_idx = top_idx[np.argsort(topic[top_idx])]  # ascending
    labels = bow_terms[top_idx]
    values = topic[top_idx]

    r = (t // cols) + 1
    c = (t % cols) + 1
    fig.add_trace(
        go.Bar(x=values, y=labels, orientation="h", name=f"Topic {t+1}", text=labels, hovertemplate="%{y}: %{x}<extra></extra>"),
        row=r, col=c
    )
    # Important: set category order per subplot + allow extra margin
    fig.update_yaxes(categoryorder="array", categoryarray=labels.tolist(), automargin=True, row=r, col=c)

# Global layout tweaks to prevent clipping
fig.update_layout(
    height=300*rows,
    title="LDA Topics — Top Terms per Topic",
    showlegend=False,
    margin=dict(l=160, r=20, t=60, b=40),  # increase left margin
)

fig.show()


iteration: 1 of max_iter: 10
iteration: 2 of max_iter: 10
iteration: 3 of max_iter: 10
iteration: 4 of max_iter: 10
iteration: 5 of max_iter: 10
iteration: 6 of max_iter: 10
iteration: 7 of max_iter: 10
iteration: 8 of max_iter: 10
iteration: 9 of max_iter: 10
iteration: 10 of max_iter: 10


LDA Topics: 100%|██████████| 10/10 [00:00<00:00, 413.17it/s]


### Text Similarity Analysis

In [39]:
def show_similar(idx=0, k=5):
    sims = cosine_similarity(X_tfidf[idx], X_tfidf).flatten()
    top_idx = sims.argsort()[-(k+1):][::-1]  # include self
    ref = imdb_data.iloc[idx]
    print("=== Reference Review ===")
    print(f"[idx={idx}] SENT={ref['sentiment']}\n", ref["review"][:700], "...\n")
    print("=== Most Similar ===")
    for j in top_idx[1:]:
        r = imdb_data.iloc[j]
        print(f"[idx={j}] sim={sims[j]:.3f} SENT={r['sentiment']}  ::  {r['review'][:140]}...")


show_similar(idx=1234, k=5)


=== Reference Review ===
[idx=1234] SENT=positive
 The 20th animated Disney classic is often criticized by many people as "mediocre" or poor in quality, but it is a great movie.<br /><br />Too bad that "The Aristocats" doesn't get the deserved credit. I personally see it as one of my favorite Disney classics.<br /><br />Despite being extremely underrated, it is one of the funniest Disney classics. It is full of hilarious (some of them, hysterical) moments.<br /><br />Edgar, the greedy butler, is the villain of the movie but he is a perfect comic relief. He's one of my favorite Disney villains because he is so funny.<br /><br />Every scene with Edgar and the hound dogs Napoleon and Lafayette chasing him are among the most hilarious you'll eve ...

=== Most Similar ===
[idx=33003] sim=0.448 SENT=positive  ::  Set in Paris in the year 1910, a retired old rich opera singer decides to give her fortune away to her beautiful cat Duchess ( voiced by Eva...
[idx=13395] sim=0.411 SENT=positive  

In [40]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=2, random_state=RANDOM_STATE)
X_2d = svd.fit_transform(X_tfidf)

vis_df = pd.DataFrame({
    "x": X_2d[:,0], "y": X_2d[:,1],
    "sentiment": imdb_data["sentiment"].values
})
fig = px.scatter(
    vis_df.sample(min(len(vis_df), 10000), random_state=RANDOM_STATE),
    x="x", y="y", color="sentiment", opacity=0.6,
    title="2D Projection of Reviews (TF-IDF → TruncatedSVD)"
)
fig.show()


### Model Performance

In [41]:
from sklearn.metrics import accuracy_score, f1_score

acc_nb = accuracy_score(y_test, pred_nb)
f1_nb  = f1_score(y_test, pred_nb)

acc_lr = accuracy_score(y_test, pred_lr)
f1_lr  = f1_score(y_test, pred_lr)

results_df = pd.DataFrame({
    "Model": ["Naive Bayes", "Logistic Regression"],
    "Accuracy": [acc_nb, acc_lr],
    "F1": [f1_nb, f1_lr]
})
display(results_df)

fig = go.Figure()
fig.add_trace(go.Bar(x=results_df["Model"], y=results_df["Accuracy"], name="Accuracy"))
fig.add_trace(go.Bar(x=results_df["Model"], y=results_df["F1"], name="F1"))
fig.update_layout(barmode="group", title="Model Performance Comparison")
fig.show()


,Model,Accuracy,F1
0,Naive Bayes,0.4988,0.567035
1,Logistic Regression,0.8516,0.890754
